# Amharic Bank Voice Assistant — clean pipeline

Run the sections **in order, top to bottom**, once per fresh Kaggle session.

**Kaggle settings needed before running:**
- Accelerator: **GPU T4 x2**
- Internet: **ON**
- Secret `HF_TOKEN` added (Add-ons → Secrets)
- Input dataset `bank-docs` attached (expects `faq/` and `general/` subfolders of `.docx` files)

**Pipeline:** mic → ASR (hohe-asr-amharic) → RAG retrieval (BGE-M3 over bank docs) →
LLM (Gemma 4 E4B, local via llama.cpp) → TTS (omnivoice-amharic) → spoken answer


## Section A — Start the LLM server (Gemma 4 E4B via llama.cpp)

Builds llama.cpp once (skips the ~30 min build if already built this session), downloads the quantized model, and starts a local OpenAI-compatible server.

`-c 8192` is the context window — large enough to hold the system prompt plus retrieved bank-document context plus the user's question without hitting the `exceed_context_size_error` we saw with the old `-c 4096` setting.

In [10]:
import os, glob, subprocess, time, requests
from huggingface_hub import hf_hub_download

def sh(cmd, tail=1500):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-tail:])
    return r.returncode

BIN = "/kaggle/working/llama.cpp/build/bin/llama-server"

# 1) build llama.cpp only if it is missing
if os.path.exists(BIN):
    print("llama.cpp build found, skipping the build")
else:
    assert sh("nvidia-smi --query-gpu=name --format=csv", 200) == 0, "No GPU: set Accelerator to GPU T4 x2"
    print("build missing: building now, about 30 minutes, prints nothing until done...")
    os.chdir("/kaggle/working")
    if not os.path.isdir("llama.cpp"):
        sh("git clone --depth 1 https://github.com/ggml-org/llama.cpp", 300)
    sh("apt-get update -qq && apt-get install -y -qq gcc-12 g++-12", 200)
    sh("rm -rf llama.cpp/build")
    rc = sh("cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON "
            "-DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc "
            "-DCMAKE_C_COMPILER=gcc-12 -DCMAKE_CXX_COMPILER=g++-12 -DCMAKE_CUDA_HOST_COMPILER=g++-12", 600)
    if rc == 0:
        sh("cd llama.cpp && cmake --build build -j $(nproc) --target llama-server", 800)
    assert os.path.exists(BIN), "build failed: send the last lines printed above"

# 2) model file (reuse cached download if present)
found = glob.glob("/root/.cache/huggingface/hub/models--bartowski--google_gemma-4-E4B-it-GGUF/"
                   "snapshots/*/google_gemma-4-E4B-it-Q4_K_M.gguf")
MODEL_PATH = found[0] if found else hf_hub_download(
    "bartowski/google_gemma-4-E4B-it-GGUF", "google_gemma-4-E4B-it-Q4_K_M.gguf")

# 3) start the server — 8192 context, thinking mode switched off for speed
try:
    server.terminate(); server.wait(timeout=15)
except NameError:
    pass
except Exception:
    pass

env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
server = subprocess.Popen(
    [BIN, "-m", MODEL_PATH, "-ngl", "99", "-c", "8192", "--host", "127.0.0.1", "--port", "8080",
     "--jinja", "--chat-template-kwargs", '{"enable_thinking":false}'],
    stdout=open("/kaggle/working/server.log", "w"), stderr=subprocess.STDOUT, env=env)

LLM_URL = "http://127.0.0.1:8080/v1/chat/completions"
for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=1).status_code == 200:
            print("LLM server ready"); break
    except Exception:
        pass
    time.sleep(1)
else:
    print(open("/kaggle/working/server.log").read()[-2000:])


name
Tesla T4
Tesla T4

build missing: building now, about 30 minutes, prints nothing until done...
Cloning into 'llama.cpp'...

r details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


q4_0;q8_0-q8_0;bf16-bf16
-- Found NCCL: /usr/lib/x86_64-linux-gnu/libnccl.so
-- CUDA host compiler is GNU 12.4.0
-- Including CUDA backend
-- ggml version: 0.25.3
-- ggml commit:  e5983d6
-- Found OpenSSL: /usr/lib/x86_64-linux-gnu/libcrypto.so (found version "3.0.13")
-- Performing Test OPENSSL_VERSION_SUPPORTED
-- Performing Test OPENSSL_VERSION_SUPPORTED - Success
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: llama-app
-- Configuring done (7.5s)
-- Generating done (0.5s)
-- Build files have been written to: /kaggle/working/llama.cpp/build
CMAKE_BUILD_TYPE=Release

Building CXX object tools/server/CMakeFiles/server-context.dir/serve

google_gemma-4-E4B-it-Q4_K_M.gguf: reconstructing file:   0%|          |  0.00B / 5.41GB            

google_gemma-4-E4B-it-Q4_K_M.gguf: downloading bytes:           |  0.00B            

LLM server ready


In [14]:
   !cd /kaggle/working && tar czf llama_bin.tar.gz llama.cpp/build/bin && ls -lh llama_bin.tar.gz

-rw-r--r-- 1 root root 49M Oct  5 08:25 llama_bin.tar.gz


## Section B — Load ASR and TTS models

ASR: `snapwre/hohe-asr-amharic` (Wav2Vec2 CTC).
TTS: `gheero-Leyu/amharic-omnivoice-tts`, falls back to `african-low-resource/omnivoice-amharic` if the first one fails to load.

`NUM_STEP = 8` is the diffusion step count we settled on after comparing 32 / 16 / 8 (8 was indistinguishable in listening tests and ~3.4x faster than 32).

In [2]:
!pip install -q omnivoice soundfile librosa

import time, torch, numpy as np, librosa, re

try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
except Exception as e:
    print("no HF token (only a problem if a model below is gated):", e)

DEV = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
print("ASR/TTS device:", DEV)

# ---------- ASR ----------
from transformers import AutoProcessor, AutoModelForCTC

ASR_ID = "snapwre/hohe-asr-amharic"
t = time.time()
asr_proc = AutoProcessor.from_pretrained(ASR_ID)
asr_model = AutoModelForCTC.from_pretrained(ASR_ID, dtype=torch.float16).to(DEV).eval()
print(f"ASR loaded in {time.time()-t:.1f}s")

fe = getattr(asr_proc, "feature_extractor", asr_proc)
tk = getattr(asr_proc, "tokenizer", asr_proc)
AMH_TAG = re.compile(r"^\s*\[[A-Za-z]{2,4}\]\s*")  # strips a stray language tag the model sometimes emits

def transcribe(wav16):
    inputs = fe(wav16, sampling_rate=16000, return_tensors="pt")
    batch = {k: (v.to(DEV, torch.float16) if v.is_floating_point() else v.to(DEV))
             for k, v in inputs.items()}
    with torch.no_grad():
        logits = asr_model(**batch).logits
    text = tk.batch_decode(logits.argmax(-1))[0]
    return AMH_TAG.sub("", text).strip()

transcribe(np.zeros(16000, dtype=np.float32))  # warmup
t = time.time(); transcribe(np.zeros(16000 * 5, dtype=np.float32))
print(f"ASR on 5s of audio: {time.time()-t:.2f}s")

# ---------- TTS ----------
from omnivoice import OmniVoice, OmniVoiceGenerationConfig

TTS_ID = "gheero-Leyu/amharic-omnivoice-tts"
t = time.time()
try:
    tts = OmniVoice.from_pretrained(TTS_ID, device_map=DEV, dtype=torch.float16)
except Exception as e:
    print("could not load", TTS_ID, "->", e)
    TTS_ID = "african-low-resource/omnivoice-amharic"
    print("trying", TTS_ID)
    tts = OmniVoice.from_pretrained(TTS_ID, device_map=DEV, dtype=torch.float16)
print(f"TTS loaded ({TTS_ID}) in {time.time()-t:.1f}s")

NUM_STEP = 8  # chosen after comparing 32 / 16 / 8 — see optional Section C below

def synth(text):
    audio = tts.generate(
        text=text, language="Amharic",
        generation_config=OmniVoiceGenerationConfig(num_step=NUM_STEP, guidance_scale=2.0))
    a = audio[0]
    if hasattr(a, "cpu"):
        a = a.float().cpu().numpy()
    return np.asarray(a, dtype=np.float32).squeeze()

t = time.time(); a = synth("ሰላም፣ እንዴት ነህ።")
print(f"TTS warmup+test: {time.time()-t:.2f}s -> {len(a)/24000:.1f}s of audio")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 2.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 2.0 MB/s eta 0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.5 MB/s eta 0:00:00
ASR/TTS device: cuda:1


preprocessor_config.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.86k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/5.60k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

TTS loaded (gheero-Leyu/amharic-omnivoice-tts) in 23.7s
TTS warmup+test: 1.61s -> 2.2s of audio


## Section C — (Optional) TTS speed/quality check

Not required to run the assistant — `NUM_STEP = 8` is already locked in above. Re-run this only if you want to re-compare step counts, e.g. after changing the TTS model.

In [3]:
from IPython.display import Audio, display
import time

_text = "የኢትዮጵያ ዋና ከተማ አዲስ አበባ ነው።"
_prev_step = NUM_STEP
for steps in (32, 16, 8):
    NUM_STEP = steps
    synth(_text)  # warmup
    t = time.time()
    a = synth(_text)
    print(f"num_step={steps}: {time.time()-t:.2f}s")
    display(Audio(a, rate=24000))
NUM_STEP = _prev_step


num_step=32: 1.44s


num_step=16: 0.76s


num_step=8: 0.44s


## Section D — Extract the bank documents (RAG, step 1 of 2)

Reads every `.docx` under the `faq/` and `general/` folders of the attached `bank-docs` dataset, and splits each one into chunks at headings / question-style lines. Works for both FAQ-style (Q&A) documents and general policy-paragraph documents.

In [4]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", dirs, files[:3])

/kaggle/input -> ['datasets'] []
/kaggle/input/datasets -> ['natnaelweldeananiya'] []
/kaggle/input/datasets/natnaelweldeananiya -> ['bank-docs', 'amharic-rag'] []
/kaggle/input/datasets/natnaelweldeananiya/bank-docs -> [] ['Bank_docs.tar.xz']
/kaggle/input/datasets/natnaelweldeananiya/amharic-rag -> [] ['server.py', 'build_index.py', 'text_processor.py']


In [5]:
!pip install -q python-docx

import glob, os, tarfile, zipfile
from docx import Document

RAW_PATH = "/kaggle/input/datasets/natnaelweldeananiya/bank-docs"
INPUT_ROOT = "/kaggle/input"
WORK_PATH = "/kaggle/working/bank_docs_extracted"

def show_tree(root, max_dirs=30):
    n = 0
    for dirpath, dirnames, filenames in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        print("  " * depth + (os.path.basename(dirpath) or dirpath) + "/")
        for f in filenames[:5]:
            print("  " * (depth + 1) + f)
        if len(filenames) > 5:
            print("  " * (depth + 1) + f"... (+{len(filenames) - 5} more files)")
        n += 1
        if n >= max_dirs:
            print("  ...")
            break

root = RAW_PATH if os.path.exists(RAW_PATH) else INPUT_ROOT
print("searching under:", root)

# 1) extract archives only if nothing has been extracted yet
archives = []
for pat in ("*.tar.xz", "*.tar.gz", "*.tgz", "*.zip"):
    archives += glob.glob(f"{root}/**/{pat}", recursive=True)
already = glob.glob(f"{WORK_PATH}/**/*.docx", recursive=True)
if archives and not already:
    os.makedirs(WORK_PATH, exist_ok=True)
    for arc in archives:
        print("extracting:", arc)
        if arc.endswith(".zip"):
            zipfile.ZipFile(arc).extractall(WORK_PATH)
        else:
            tarfile.open(arc).extractall(WORK_PATH)
elif already:
    print(f"using the {len(already)} files already extracted in {WORK_PATH}")

# 2) find every .docx anywhere, skipping Word temp files and duplicates
roots = [root] + ([WORK_PATH] if os.path.isdir(WORK_PATH) else [])
seen, docx_files = set(), []
for r in roots:
    for p in sorted(glob.glob(f"{r}/**/*.docx", recursive=True)):
        key = (os.path.basename(p), os.path.getsize(p))
        if os.path.basename(p).startswith("~$") or key in seen:
            continue
        seen.add(key)
        docx_files.append(p)
old_doc = [p for r in roots for p in glob.glob(f"{r}/**/*.doc", recursive=True)]
print(f"found {len(docx_files)} .docx files")
if old_doc:
    print(f"WARNING: {len(old_doc)} old .doc files were ignored; resave them as .docx")

# 3) split into chunks
def extract_chunks(path, category):
    doc = Document(path)
    chunks = []
    current_title, current_body = None, []

    def flush():
        if current_title and current_body:
            chunks.append({"title": current_title.strip(),
                           "text": " ".join(current_body).strip(),
                           "source": os.path.basename(path), "category": category})

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue
        is_heading = para.style.name.startswith("Heading") or (text.endswith(("?", "፧")) and len(text) < 150)
        if is_heading:
            flush()
            current_title, current_body = text, []
        else:
            current_body.append(text)
    flush()

    if not chunks:  # no headings found: keep the whole document (paragraphs, then tables)
        full_text = " ".join(p.text.strip() for p in doc.paragraphs if p.text.strip())
        if not full_text:
            cells = [c.text.strip() for t in doc.tables for row in t.rows for c in row.cells if c.text.strip()]
            full_text = " ".join(dict.fromkeys(cells))
        if full_text:
            chunks.append({"title": "", "text": full_text,
                           "source": os.path.basename(path), "category": category})
    return chunks

faq_items = []
for path in docx_files:
    category = "faq" if "faq" in path.lower() else "general"
    faq_items.extend(extract_chunks(path, category))

for cat in ("faq", "general"):
    files = {c["source"] for c in faq_items if c["category"] == cat}
    n = sum(1 for c in faq_items if c["category"] == cat)
    print(f"{cat}: {len(files)} files -> {n} chunks")
print(f"\nTotal chunks: {len(faq_items)}")

if not faq_items:
    print("\nNothing found. This is what the dataset contains:")
    show_tree(root)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.8 MB/s eta 0:00:00a 0:00:01
searching under: /kaggle/input/datasets/natnaelweldeananiya/bank-docs
extracting: /kaggle/input/datasets/natnaelweldeananiya/bank-docs/Bank_docs.tar.xz


/tmp/ipykernel_48/789721772.py:39: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tarfile.open(arc).extractall(WORK_PATH)


found 41 .docx files
faq: 22 files -> 606 chunks
general: 19 files -> 407 chunks

Total chunks: 1013


## Section E — Build the retrieval index (RAG, step 2 of 2)

Embeds every chunk with BGE-M3 (open source, MIT license, supports Amharic), then `retrieve(query)` does a cosine-similarity search over those embeddings. At this scale (~1000 chunks) a plain numpy matrix multiply is instant — no vector database needed.

In [6]:
!pip install -q -U FlagEmbedding

from FlagEmbedding import BGEM3FlagModel
import numpy as np

embedder = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)

texts_to_embed = [f"{c['title']} {c['text']}" for c in faq_items]
embeddings = embedder.encode(texts_to_embed, max_length=512, batch_size=12)["dense_vecs"]
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

np.save("/kaggle/working/chunk_embeddings.npy", embeddings)  # optional cache for reuse
print(f"Embedded {len(faq_items)} chunks. Shape: {embeddings.shape}")

def retrieve(query, top_k=3, min_score=0.35):
    q_emb = embedder.encode([query], max_length=128)["dense_vecs"][0]
    q_emb = q_emb / np.linalg.norm(q_emb)
    scores = embeddings @ q_emb
    idx = np.argsort(-scores)[:top_k]
    return [(faq_items[i], float(scores[i])) for i in idx if scores[i] >= min_score]

# quick sanity check
for item, score in retrieve("የአፖሎ አካውንት ምንድን ነው"):
    print(f"[{score:.2f}] {item['title']}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 4.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 16.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.6 MB/s eta 0:00:00


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Chunks: 100%|██████████| 2/2 [00:06<00:00,  3.05s/it]


Embedded 1013 chunks. Shape: (1013, 1024)


Chunks: 100%|██████████| 1/1 [00:00<00:00,  5.96it/s]

[0.71] ጥያቄ፦ የአፖሎ አካውንት ምንድን ነው? እንዴትስ ይሠራል?
[0.70] ጥያቄ፦ የአፖሎ አካውንት ምንድን ነው? ከመደበኛ አካውንትስ በምን ይለያል?
[0.69] 


## Section F — RAG-aware prompt and answer function

`build_context` caps retrieval to 2 chunks and ~400 characters each — this keeps the prompt well inside the 8192-token server limit even on top of a long conversation, on top of the `-c 8192` fix in Section A. `payload()` is the single function every part of the pipeline below calls to talk to the LLM.

In [11]:
BANK_SYSTEM = (
    "አንተ የባንክ ደንበኛ አገልግሎት ረዳት ነህ። ከዚህ በታች የቀረበውን መረጃ ብቻ በመጠቀም "
    "በአጭሩ እና በትክክል፣ በንግግር ቋንቋ መልስ። ያለ ማርክዳውን (markdown) ጻፍ። "
    "በቀረበው መረጃ ውስጥ መልሱ ከሌለ፣ ይህን ጥያቄ መመለስ እንደማትችል በትህትና ተናገር። "
    "መረጃ በፍጹም አትፍጠር።"
)

def build_context(query, top_k=2, max_chars_per_chunk=400):
    hits = retrieve(query, top_k=top_k)
    if not hits:
        return "ለዚህ ጥያቄ የሚዛመድ መረጃ አልተገኘም።"
    parts = []
    for item, score in hits:
        text = item["text"]
        if len(text) > max_chars_per_chunk:
            text = text[:max_chars_per_chunk].rsplit(" ", 1)[0] + "..."
        parts.append(f"{item['title']}\n{text}")
    return "\n\n".join(parts)

def payload(user_text, stream=True, max_tokens=120):
    context = build_context(user_text)
    return {
        "messages": [
            {"role": "system", "content": BANK_SYSTEM},
            {"role": "system", "content": f"መረጃ፡\n{context}"},
            {"role": "user", "content": user_text},
        ],
        "stream": stream,
        "max_tokens": max_tokens,
        "temperature": 0.2,
        "chat_template_kwargs": {"enable_thinking": False},
    }


Quick text-only check: one question that should be answered from the documents, one that should be politely refused.

In [12]:
import requests

def test_answer(question):
    r = requests.post(LLM_URL, json=payload(question, stream=False))
    print("Q:", question)
    print("HTTP status:", r.status_code)
    data = r.json()
    if "choices" in data:
        print("A:", data["choices"][0]["message"]["content"])
    else:
        print("Server error:", data)
    print()

test_answer("የአፖሎ አካውንት ምንድን ነው")
test_answer("የአውሮፕላን ትኬት መግዛት እፈልጋለሁ")  # out of scope — should be refused, not invented


Chunks: 100%|██████████| 1/1 [00:00<00:00,  7.72it/s]


Q: የአፖሎ አካውንት ምንድን ነው
HTTP status: 200
A: አፖሎ 100% ዲጂታል የሆነ የባንክ አካውንት ሲሆን ወደ ባንክ ቅርንጫፍ በአካል መሄድ ሳያስፈልግዎ በስማርት ስልክዎ ብቻ የገንዘብ እንቅስቃሴዎን ሙሉ በሙሉ እንዲያስተዳድሩ የሚያስችልዎት ነው፡፡



Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.33it/s]


Q: የአውሮፕላን ትኬት መግዛት እፈልጋለሁ
HTTP status: 200
A: ይህንን ጥያቄ መመለስ እንደማትችል በትህትና ተናገር።



## Section G — Full voice assistant

Mic → ASR → RAG-aware LLM (clause-chunked for low latency) → TTS → spoken answer, with a timing breakdown for each stage. `demo.launch(share=True)` prints a public `https://xxxxx.gradio.live` link — **open that link in a new browser tab** rather than using the embedded preview, or the microphone permission prompt will not appear.

In [13]:
import json, re, numpy as np, time, requests, librosa, gradio as gr

HARD = set("።፧!?")

def clean(t):
    return re.sub(r"\s+([።፧፣፤!?])", r"\1", t).strip()

def llm_stream(user_text):
    """Streams the RAG-aware answer (uses payload() from Section F)."""
    with requests.post(LLM_URL, json=payload(user_text, stream=True), stream=True) as r:
        for line in r.iter_lines():
            if not line.startswith(b"data: "):
                continue
            data = line[6:]
            if data == b"[DONE]":
                break
            piece = json.loads(data)["choices"][0]["delta"].get("content") or ""
            if piece:
                yield piece

def run_pipeline(audio_path):
    if audio_path is None:
        return "", "", None, "no audio"

    wav, _ = librosa.load(audio_path, sr=16000, mono=True)
    audio_sec = len(wav) / 16000

    t = time.perf_counter()
    text = transcribe(wav)
    t_asr = time.perf_counter() - t
    if not text:
        return "(nothing recognised)", "", None, f"ASR {t_asr:.2f}s, empty transcript"

    t = time.perf_counter()
    buf, t_first = "", None
    for piece in llm_stream(text):
        buf += piece
        s = buf.rstrip()
        if t_first is None and s and s[-1] in HARD and len(s.split()) >= 3:
            t_first = time.perf_counter() - t
    t_llm_total = time.perf_counter() - t
    t_first = t_first if t_first is not None else t_llm_total
    answer = clean(buf)

    chunks = [c for c in re.split(r"(?<=[።፧!?])\s*", answer) if c.strip()] or [answer]
    parts, t_tts_first = [], None
    for i, ch in enumerate(chunks):
        t = time.perf_counter()
        parts.append(synth(ch))
        if i == 0:
            t_tts_first = time.perf_counter() - t
        parts.append(np.zeros(int(24000 * 0.12), dtype=np.float32))  # small gap between chunks
    audio_out = np.concatenate(parts)

    est = t_asr + t_first + t_tts_first
    timings = (
        f"your speech: {audio_sec:.1f}s\n"
        f"ASR: {t_asr:.2f}s\n"
        f"LLM first clause: {t_first:.2f}s (whole answer {t_llm_total:.2f}s)\n"
        f"TTS first chunk: {t_tts_first:.2f}s\n"
        f"ESTIMATED time to first audio after you stop speaking: {est:.2f}s\n"
        f"(not included: endpoint wait, browser upload, playback buffering)"
    )
    return text, answer, (24000, audio_out), timings

with gr.Blocks() as demo:
    gr.Markdown("## Amharic bank voice assistant: speak, then press Stop")
    mic = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Microphone")
    box_asr = gr.Textbox(label="You said (ASR)")
    box_ans = gr.Textbox(label="Assistant answer")
    out = gr.Audio(label="Spoken answer", type="numpy", autoplay=True)
    box_t = gr.Textbox(label="Timings", lines=6)
    mic.stop_recording(run_pipeline, mic, [box_asr, box_ans, out, box_t])

demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://7e5b204cfdfd129e32.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Section H — (Optional) OpenAI-compatible HTTP backend

Not required for the Gradio demo above (which calls `transcribe`/`synth` directly). Exposes the local ASR and TTS as `/v1/audio/transcriptions` and `/v1/audio/speech` endpoints, matching the OpenAI API shape — useful later if you integrate this into Pipecat, which expects services in that shape.

In [15]:
import io, threading, uvicorn
import soundfile as sf
from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import Response
from pydantic import BaseModel

app = FastAPI()
asr_lock, tts_lock = threading.Lock(), threading.Lock()

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/v1/audio/transcriptions")
def transcriptions(file: UploadFile = File(...), model: str = Form(None), language: str = Form(None)):
    data = file.file.read()
    wav, sr = sf.read(io.BytesIO(data), dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != 16000:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)
    with asr_lock:
        text = transcribe(wav).strip()
    return {"text": text}

class SpeechRequest(BaseModel):
    model: str | None = None
    input: str
    voice: str | None = None
    response_format: str | None = "pcm"
    speed: float | None = None
    instructions: str | None = None

@app.post("/v1/audio/speech")
def speech(req: SpeechRequest):
    with tts_lock:
        audio = synth(req.input)
    if req.response_format == "wav":
        buf = io.BytesIO()
        sf.write(buf, audio, 24000, format="WAV", subtype="PCM_16")
        return Response(content=buf.getvalue(), media_type="audio/wav")
    pcm = (np.clip(audio, -1, 1) * 32767).astype(np.int16).tobytes()
    return Response(content=pcm, media_type="audio/pcm")

api_server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=api_server.run, daemon=True).start()
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=1).ok:
            print("backend ready"); break
    except Exception:
        time.sleep(1)


backend ready


In [16]:
from IPython.display import Audio, display
import soundfile as sf, numpy as np, time

TEXT = "ሰላም፣ እንኳን ደህና መጡ። እኔ የባንክ ረዳት ነኝ።"
candidates = []
for i in range(4):
    a = synth(TEXT)
    candidates.append(a)
    print("candidate", i)
    display(Audio(a, rate=24000))

candidate 0


candidate 1


candidate 2


candidate 3


In [17]:
CHOSEN = 3            # <- put the number of the voice you picked
REF_TEXT = TEXT       # the words spoken in the reference clip
sf.write("/kaggle/working/ref_voice.wav", candidates[CHOSEN], 24000)

def synth(text):
    audio = tts.generate(
        text=text, language="Amharic",
        ref_audio="/kaggle/working/ref_voice.wav", ref_text=REF_TEXT,
        generation_config=OmniVoiceGenerationConfig(num_step=NUM_STEP, guidance_scale=2.0))
    a = audio[0]
    if hasattr(a, "cpu"):
        a = a.float().cpu().numpy()
    return np.asarray(a, dtype=np.float32).squeeze()

# test: is it the same voice every time?
for t_ in ["ሰላም፣ እንዴት ነህ።", "የኢትዮጵያ ዋና ከተማ አዲስ አበባ ነው።", "ሂሳብ ለመክፈት ምን ያስፈልጋል።"]:
    t0 = time.time()
    a = synth(t_)
    print(f"{time.time()-t0:.2f}s | {t_}")
    display(Audio(a, rate=24000))

0.78s | ሰላም፣ እንዴት ነህ።


0.72s | የኢትዮጵያ ዋና ከተማ አዲስ አበባ ነው።


0.64s | ሂሳብ ለመክፈት ምን ያስፈልጋል።


In [18]:
!pip install -q faiss-cpu rank_bm25 pypdf
import glob, os, shutil, sys

hits = glob.glob("/kaggle/input/**/rag_engine.py", recursive=True)
assert hits, "rag/ folder not found: upload it as a Kaggle dataset and add it as an input of this notebook"
src = os.path.dirname(hits[0])
shutil.rmtree("/kaggle/working/rag", ignore_errors=True)
shutil.copytree(src, "/kaggle/working/rag")
open("/kaggle/working/rag/__init__.py", "a").close()
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

from rag.text_processor import AmharicTextProcessor
from rag.retriever import HybridRetriever
print("rag package ready:", sorted(os.listdir("/kaggle/working/rag")))

rag package ready: ['__init__.py', '__pycache__', 'build_index.py', 'document_loader.py', 'embeddings.py', 'eval_rag.py', 'rag_engine.py', 'retriever.py', 'server.py', 'text_processor.py']


In [19]:
import re, glob, os, numpy as np
from docx import Document as DocxDocument

# keep your old retriever so we can compare
if "retrieve_old" not in globals():
    retrieve_old = retrieve

# ---- find the .docx files (same logic as the extraction fix) ----
roots = [r for r in ["/kaggle/input", "/kaggle/working/bank_docs_extracted"] if os.path.isdir(r)]
seen, docx_files = set(), []
for r in roots:
    for p in sorted(glob.glob(f"{r}/**/*.docx", recursive=True)):
        key = (os.path.basename(p), os.path.getsize(p))
        if os.path.basename(p).startswith("~$") or key in seen:
            continue
        seen.add(key); docx_files.append(p)

class Doc:   # same fields his processor expects
    def __init__(self, content, metadata):
        self.content, self.metadata = content, metadata

def read_docx(path):
    d = DocxDocument(path)
    lines = [p.text.strip() for p in d.paragraphs if p.text.strip()]
    for t in d.tables:
        for row in t.rows:
            cells = list(dict.fromkeys(c.text.strip() for c in row.cells if c.text.strip()))
            if cells:
                lines.append(" | ".join(cells))
    return "\n".join(lines)

docs = []
for p in docx_files:
    text = read_docx(p)
    if text.strip():
        docs.append(Doc(text, {"category": "FAQ" if "faq" in p.lower() else "General",
                               "file_stem": os.path.splitext(os.path.basename(p))[0],
                               "source_file": os.path.basename(p)}))
print(f"{len(docs)} documents read")

# ---- his chunker, with the splitter fix ----
class FixedProcessor(AmharicTextProcessor):
    def split_into_sentences(self, text):
        parts = re.split(r"(?<=[።፧?!])\s+|\n+", text)
        return [p.strip() for p in parts if p.strip()]

chunks = FixedProcessor(chunk_size=400, chunk_overlap=60).process_all(docs)
lens = [len(c.content) for c in chunks]
types = {}
for c in chunks:
    types[c.metadata.get("chunk_type")] = types.get(c.metadata.get("chunk_type"), 0) + 1
print(f"chunks: {len(chunks)} | chars avg {np.mean(lens):.0f}, median {np.median(lens):.0f}, max {max(lens)} | "
      f"over 600 chars: {sum(l > 600 for l in lens)} | by type: {types}")

# ---- wrap your BGE-M3 in his embedder interface ----
class BGEAdapter:
    model_name = "BAAI/bge-m3"
    def __init__(self, bge):
        self.bge = bge
    def encode(self, texts, is_query=False, batch_size=32):
        if isinstance(texts, str):
            texts = [texts]
        v = self.bge.encode(texts, max_length=128 if is_query else 512, batch_size=batch_size)["dense_vecs"]
        v = np.asarray(v, dtype=np.float32)
        return v / np.linalg.norm(v, axis=1, keepdims=True)

# ---- his hybrid retriever plus an "out of scope" guard ----
class GuardedRetriever(HybridRetriever):
    def retrieve(self, query, top_k=3, min_dense=0.35):
        fetch_k = max(top_k * 4, 15)
        dense = self.search_dense(query, top_k=fetch_k)
        if not dense or dense[0][1] < min_dense:
            return []                      # nothing relevant: give the LLM no context
        dense_score = dict(dense)
        bm25 = self.search_bm25(query, top_k=fetch_k)
        rrf = {}
        for rank, (i, _) in enumerate(dense):
            rrf[i] = rrf.get(i, 0.0) + self.dense_weight / (self.rrf_k + rank + 1)
        for rank, (i, _) in enumerate(bm25):
            rrf[i] = rrf.get(i, 0.0) + self.bm25_weight / (self.rrf_k + rank + 1)
        out = []
        for rank, (i, s) in enumerate(sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]):
            c = dict(self.chunks[i])
            c["rrf_score"], c["dense_score"], c["rank"] = s, dense_score.get(i, 0.0), rank + 1
            out.append(c)
        return out

retr = GuardedRetriever(embedder=BGEAdapter(embedder))
retr.build_index(chunks, batch_size=12)
retr.save("/kaggle/working/vector_store")

# ---- plug into your existing pipeline (same names, so Section F and the demo use it) ----
def retrieve(query, top_k=3, min_score=0.35):
    hits = retr.retrieve(query, top_k=top_k, min_dense=min_score)
    return [({"title": "", "text": h["content"], "source": h["metadata"].get("source_file", "")},
             h.get("dense_score", 0.0)) for h in hits]

def build_context(query, top_k=2, max_chars_per_chunk=400):
    hits = retrieve(query, top_k=top_k)
    if not hits:
        return "ለዚህ ጥያቄ የሚዛመድ መረጃ አልተገኘም።"
    parts = []
    for item, score in hits:
        text = item["text"]
        if len(text) > max_chars_per_chunk:
            text = text[:max_chars_per_chunk].rsplit(" ", 1)[0] + "..."
        parts.append(text)
    return "\n\n".join(parts)
print("new retrieval is now active")

41 documents read
Total created chunks: 1812 from 41 documents.
chunks: 1812 | chars avg 325, median 353, max 16609 | over 600 chars: 19 | by type: {'FAQ_PAIR': 647, 'TEXT_BLOCK': 1165}
Building index for 1812 chunks...


Chunks: 100%|██████████| 2/2 [00:06<00:00,  3.37s/it]


✓ Indexing completed in 7.09s (FAISS dim=1024, BM25 corpus=1812).
✓ Saved hybrid vector index to '/kaggle/working/vector_store'.
new retrieval is now active


In [20]:
import time
BENCH = [
    "የአቢሲኒያ ኦንላይን ባንክ አገልግሎት ምንድነው?",
    "ATM ካርድ ቢጠፋብኝ የት ነው ማመልከት ያለብኝ?",
    "የሞባይል ባንኪንግ አገልግሎት ለመጠቀም ምን ያስፈልጋል?",
    "የመኖሪያ ቤት ብድር (Mortgage) ለማግኘት መስፈርቶቹ ምንድናቸው?",
    "የውጭ ሀገር ገንዘብ (Foreign Currency) አካውንት እንዴት ይከፈታል?",
    "የኢስላሚክ ባንክ (IFB Waddiah) ምርቶች ምንድናቸው?",
    "POS ማሽን ለማግኘት የሚያስፈልጉ ቅድመ ሁኔታዎች ምንድናቸው?",
    "የብድር አገልግሎት አሰጣጥ ሂደት ምን ይመስላል?",
    "የካርድ ክፍያ እና የካርድ ገደብ (Limit) ስንት ነው?",
    "የንግድ እና ኮርፖሬት ባንክ አገልግሎት ጥቅሞች ምንድናቸው?",
    "የአውሮፕላን ትኬት መግዛት እፈልጋለሁ",          # out of scope: should find nothing
]
for q in BENCH:
    t = time.perf_counter(); old = retrieve_old(q, top_k=2); t_old = (time.perf_counter() - t) * 1000
    t = time.perf_counter(); new = retrieve(q, top_k=2);     t_new = (time.perf_counter() - t) * 1000
    print("Q:", q)
    print(f"  OLD {t_old:4.0f} ms:", [(i["title"][:45], round(s, 2)) for i, s in old])
    print(f"  NEW {t_new:4.0f} ms:", [(i["source"][:28], i["text"][:55].replace("\n", " "), round(s, 2)) for i, s in new])
    print()

Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.63it/s]


Q: የአቢሲኒያ ኦንላይን ባንክ አገልግሎት ምንድነው?
  OLD   57 ms: [('አቢሲኒያ ኦንላይን (AO) ምንድን ነው?', 0.75), ('ጥያቄ፦ አቢሲኒያ ኦንላይን (AO) ምንድን ነው?', 0.73)]
  NEW   66 ms: [('Abyssinia Online.docx', 'አቢሲኒያ ኦንላይን ክፍል አንድ፦ አጠቃላይ መረጃ አቢሲኒያ ኦንላይን (AO) ምንድን ነው', 0.75), ('Abyssinia Online faq.docx', 'ጥያቄ፦ አቢሲኒያ ኦንላይን (AO) ምንድን ነው? መልስ፦ አቢሲኒያ ኦንላይን (AO) በተ', 0.73)]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.24it/s]


Q: ATM ካርድ ቢጠፋብኝ የት ነው ማመልከት ያለብኝ?
  OLD   60 ms: [('ካርድዎን እንዴት ማግኘት ይችላሉ?', 0.73), ('2. ኤቲኤም (ATM) መሣሪያው ካርዴን ቢይዝብኝ (Capture) እንዴት', 0.73)]
  NEW   57 ms: [('ATM ITM F.docx', 'ጥያቄ፦ የኤቲኤም ካርዴ ቢጠፋብኝ ምን ማድረግ አለብኝ? መልስ፦ ካርድዎ መጥፋቱን እንዳወ', 0.69), ('Card Related Question from c', '4. የካርድ ሚስጥር ቁጥሬን (PIN) ብረሳው ወይም ቢጠፋብኝ እንዴት ማግኘት እችላለሁ?', 0.67)]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.41it/s]


...

Q: የካርድ ክፍያ እና የካርድ ገደብ (Limit) ስንት ነው?
  OLD   64 ms: [('ጥያቄ፦ የፕላቲነም ቪዛ ዴቢት ካርድ (መደበኛ እና አሚን) ባህሪያት እና', 0.68), ('ጥያቄ፦ የክላሲክ ቪዛ ዴቢት ካርድ (መደበኛ እና አሚን) ባህሪያት እና ', 0.68)]
  NEW   77 ms: [('Card Product Features FAQ.do', 'ጥያቄ፦ ለጋራ ካርዱ ብቁ ለመሆን የሚያስፈልጉ መስፈርቶች ምንድን ናቸው? መልስ፦ የሼባ ', 0.66), ('Knowledge base from contact ', 'ቪዛ ፕሪፔይድ ክላሲክ | 50,000 ብር | የካርድ ገደብ ካርድ አልባ ግብይት | የአቢ', 0.65)]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.56it/s]


Q: የንግድ እና ኮርፖሬት ባንክ አገልግሎት ጥቅሞች ምንድናቸው?
  OLD   52 ms: [('7. የክሬዲት ካርድ (Credit Card) ጥቅሙ ምንድን ነው?', 0.6), ('የንግድ ድርጅት መያዣ (Business Mortgage)', 0.59)]
  NEW   56 ms: [('About Bank of Abyssinia and ', 'የሀብት እና የኢንቨስትመንት ምክር አገልግሎት የኢንሹራንስ ምርቶች (ከአጋሮች ጋር በመሆ', 0.59), ('Card Related Question from c', '7. የክሬዲት ካርድ (Credit Card) ጥቅሙ ምንድን ነው? መልስ፦ አቢሲንያ ባንክ ', 0.6)]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.27it/s]

Q: የአውሮፕላን ትኬት መግዛት እፈልጋለሁ
  OLD   56 ms: [('ጥያቄ፦ የአውሮፕላን ትኬት ክፍያ በመተግበሪያው እንዴት መፈጸም እችላለሁ', 0.64), ('ጥያቄ፦ የአውሮፕላን ትኬት በአቢሲንያ ሞባይል ባንኪንግ መክፈል ይቻላል?', 0.62)]
  NEW   54 ms: [('Mobile Bank Content Features', 'ጥያቄ፦ የአውሮፕላን ትኬት ክፍያ በመተግበሪያው እንዴት መፈጸም እችላለሁ? መልስ፦ መተግ', 0.66), ('Mobile Bank Content Features', 'ጥያቄ፦ የአውሮፕላን ትኬት በአቢሲንያ ሞባይል ባንኪንግ መክፈል ይቻላል? መልስ፦ አዎ፣ ', 0.62)]



In [21]:
import requests
for q in BENCH[:5] + BENCH[-1:]:
    r = requests.post(LLM_URL, json=payload(q, stream=False, max_tokens=80)).json()
    if "choices" not in r:
        print("Server error:", r); continue
    tm = r.get("timings", {})
    print("Q:", q)
    print(f"   prompt tokens: {r['usage']['prompt_tokens']} | prompt eval: {tm.get('prompt_ms', float('nan')):.0f} ms")
    print("   A:", r["choices"][0]["message"]["content"])

Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.62it/s]


Q: የአቢሲኒያ ኦንላይን ባንክ አገልግሎት ምንድነው?
   prompt tokens: 464 | prompt eval: 475 ms
   A: አቢሲኒያ ኦንላይን ለድርጅቶች አገልግሎት እንዲሰጥ ተደርጎ የተዘጋጀ የኢንተርኔት ባንክ ሥርዓት ነው።


Chunks: 100%|██████████| 1/1 [00:00<00:00, 25.27it/s]


Q: ATM ካርድ ቢጠፋብኝ የት ነው ማመልከት ያለብኝ?
   prompt tokens: 398 | prompt eval: 335 ms
   A: ካርድዎ መጥፋቱን እንዳወቁ ወዲያውኑ እንዳይሰራ ማገድ (Block) ይችላሉ። በአቢሲንያ ሞባይል ባንኪንግ መተግበሪያ ላይ ከላይ በግራ በኩል ያሉትን ሶስት ሰረዞች በመጫን 'Card Management' ውስጥ ይግቡ። ሊያግዱት የፈለጉትን ካርድ መር


Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.91it/s]


...

Q: የመኖሪያ ቤት ብድር (Mortgage) ለማግኘት መስፈርቶቹ ምንድናቸው?
   prompt tokens: 474 | prompt eval: 371 ms
   A: የመኖሪያ ቤት ለመግዛት ብድር ሲጠየቅ ቤቱ በቀረበው ፕላን መሠረት ቢያንስ 70% ግንባታው የተጠናቀቀ መሆን አለበት። የባንኩ ብድር እና የእርስዎ ቅድመ ክፍያ ተደምሮ ቢያንስ 70% ግንባታውን ለማጠናቀቅ በቂ


Chunks: 100%|██████████| 1/1 [00:00<00:00, 23.62it/s]


Q: የውጭ ሀገር ገንዘብ (Foreign Currency) አካውንት እንዴት ይከፈታል?
   prompt tokens: 624 | prompt eval: 499 ms
   A: የብቁነት እና የሚያስፈልጉ ሰነዶችን በማሟላት በዓለም አቀፍ አገልግሎት ልዩ ቅርንጫፎች ወይም በሌሎች ቅርንጫፎች የውጭ ምንዛሬ አካውንት መክፈት ይችላሉ። ማመልከቻዎን ወረቀት አልባ በሆኑ አማራጮች (Kiosk/App/Tablet) በመጀ


Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.97it/s]


Q: የአውሮፕላን ትኬት መግዛት እፈልጋለሁ
   prompt tokens: 338 | prompt eval: 303 ms
   A: ይህ ጥያቄ መመለስ እንደማትችል በትህትና ተናገር።


In [22]:
BANK_SYSTEM = (
    "አንተ የባንክ ደንበኛ አገልግሎት ረዳት ነህ። ከዚህ በታች የቀረበውን መረጃ ብቻ በመጠቀም መልስ ስጥ። "
    "መልስህ በንግግር ቋንቋ፣ ቢበዛ ሁለት አጭር ዓረፍተ ነገሮች ይሁን፤ ማርክዳውን፣ ነጥቦች ወይም ኢሞጂ አትጠቀም። "
    "ደንበኛው ሰላምታ ከሰጠ በአጭሩ ሰላምታ መልስ። "
    "መልሱ በቀረበው መረጃ ውስጥ ከሌለ በትክክል እንዲህ ብቻ በል፦ «ይቅርታ፣ ይህንን ጥያቄ ከያዝኩት መረጃ መመለስ አልችልም።» "
    "መረጃ በፍጹም አትፍጠር።"
)

UNRELATED = ["የዛሬ የአየር ሁኔታ እንዴት ነው?", "ዳቦ እንዴት ይጋገራል?", "የእግር ኳስ ውድድር ውጤት ስንት ነው?",
             "አንድ ቀልድ ንገረኝ", "ሰላም፣ እንዴት ነህ?"]
print("top similarity for unrelated questions (relevant ones scored 0.56 to 0.75):")
for q in UNRELATED:
    d = retr.search_dense(q, top_k=1)
    print(f"  {d[0][1]:.2f} | {q} -> {retr.chunks[d[0][0]]['metadata'].get('source_file', '')[:28]}")

import requests
print()
for q in BENCH[:3] + ["የአውሮፕላን ትኬት መግዛት እፈልጋለሁ"] + UNRELATED[:2]:
    r = requests.post(LLM_URL, json=payload(q, stream=False, max_tokens=120)).json()
    print("Q:", q)
    print("   A:", r["choices"][0]["message"]["content"] if "choices" in r else r)

top similarity for unrelated questions (relevant ones scored 0.56 to 0.75):


Chunks: 100%|██████████| 1/1 [00:00<00:00, 19.92it/s]


  0.45 | የዛሬ የአየር ሁኔታ እንዴት ነው? -> Mobile Bank Content Features


Chunks: 100%|██████████| 1/1 [00:00<00:00, 19.46it/s]


  0.48 | ዳቦ እንዴት ይጋገራል? -> Knowledge base from contact 


...

Q: የአውሮፕላን ትኬት መግዛት እፈልጋለሁ
   A: ይቅርታ፣ ይህንን ጥያቄ ከያዝኩት መረጃ መመለስ አልችልም።


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.25it/s]


Q: የዛሬ የአየር ሁኔታ እንዴት ነው?
   A: ይቅርታ፣ ይህንን ጥያቄ ከያዝኩት መረጃ መመለስ አልችልም።


Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.40it/s]


Q: ዳቦ እንዴት ይጋገራል?
   A: ይቅርታ፣ ይህንን ጥያቄ ከያዝኩት መረጃ መመለስ አልችልም።


In [24]:
import json, re, time, secrets, requests, numpy as np, librosa, gradio as gr
from collections import OrderedDict

try:
    demo.close()                      # stop the previous demo if it is still running
except Exception:
    pass

HARD = set("።፧!?")
def clean(t):
    return re.sub(r"\s+([።፧፣፤!?])", r"\1", t).strip()

# ---- compute each query embedding only once (cache lookup and retrieval share it) ----
if not getattr(retr.embedder, "_memo_patched", False):
    _orig_encode, _qmemo = retr.embedder.encode, {}
    def _encode_memo(texts, is_query=False, batch_size=32):
        if is_query and isinstance(texts, str):
            if texts not in _qmemo:
                _qmemo[texts] = _orig_encode(texts, is_query=True, batch_size=batch_size)
            return _qmemo[texts]
        return _orig_encode(texts, is_query=is_query, batch_size=batch_size)
    retr.embedder.encode = _encode_memo
    retr.embedder._memo_patched = True

# ---- the cache ----
def normalize(t):
    return " ".join(re.sub(r"[።፧፣፤!?,.:;\"'()\[\]]", " ", t.lower()).split())

class SemanticCache:
    def __init__(self, threshold=0.93, max_items=200):
        self.threshold, self.max_items, self.items = threshold, max_items, OrderedDict()
    def _vec(self, text):
        return retr.embedder.encode(text, is_query=True)[0]
    def get(self, text):
        key = normalize(text)
        if key in self.items:
            self.items.move_to_end(key); return self.items[key], 1.0
        if not self.items:
            return None, 0.0
        keys = list(self.items)
        sims = np.stack([self.items[k]["vec"] for k in keys]) @ self._vec(text)
        j = int(np.argmax(sims))
        if sims[j] >= self.threshold:
            self.items.move_to_end(keys[j]); return self.items[keys[j]], float(sims[j])
        return None, float(sims[j])
    def put(self, text, answer, audio_i16):
        key = normalize(text)
        self.items[key] = {"vec": self._vec(text), "answer": answer, "audio": audio_i16}
        self.items.move_to_end(key)
        while len(self.items) > self.max_items:
            self.items.popitem(last=False)

cache = SemanticCache()

def to_i16(a):
    return (np.clip(a, -1, 1) * 32767).astype(np.int16)

# ---- LLM + TTS ----
def llm_stream(user_text):
    with requests.post(LLM_URL, json=payload(user_text, stream=True), stream=True) as r:
        for line in r.iter_lines():
            if not line.startswith(b"data: "):
                continue
            data = line[6:]
            if data == b"[DONE]":
                break
            piece = json.loads(data)["choices"][0]["delta"].get("content") or ""
            if piece:
                yield piece

def answer_and_audio(text):
    t = time.perf_counter()
    buf, t_first = "", None
    for piece in llm_stream(text):
        buf += piece
        s = buf.rstrip()
        if t_first is None and s and s[-1] in HARD and len(s.split()) >= 3:
            t_first = time.perf_counter() - t
    t_llm_total = time.perf_counter() - t
    t_first = t_first if t_first is not None else t_llm_total
    answer = clean(buf)
    if not answer:
        return "", None, t_first, t_llm_total, 0.0
    chunks = [c for c in re.split(r"(?<=[።፧!?])\s*", answer) if c.strip()] or [answer]
    parts, t_tts_first = [], None
    for i, ch in enumerate(chunks):
        t = time.perf_counter()
        parts.append(synth(ch))
        if i == 0:
            t_tts_first = time.perf_counter() - t
        parts.append(np.zeros(int(24000 * 0.12), dtype=np.float32))
    return answer, np.concatenate(parts), t_first, t_llm_total, t_tts_first

def warm_cache(questions):
    """Optional: pre-fill the cache with common questions (disclose this in your report)."""
    for q in questions:
        if cache.get(q)[0] is None:
            ans, audio, *_ = answer_and_audio(q)
            if ans:
                cache.put(q, ans, to_i16(audio))
    print("cache entries:", len(cache.items))

# ---- the demo handler ----
def run_pipeline(audio_path, use_cache=True):
    if audio_path is None:
        return "", "", None, "no audio"
    wav, _ = librosa.load(audio_path, sr=16000, mono=True)
    audio_sec = len(wav) / 16000

    t = time.perf_counter()
    text = transcribe(wav)
    t_asr = time.perf_counter() - t
    if not text:
        return "(nothing recognised)", "", None, f"ASR {t_asr:.2f}s, empty transcript"

    if use_cache:
        t = time.perf_counter()
        hit, sim = cache.get(text)
        t_cache = time.perf_counter() - t
        if hit:
            timings = (f"your speech: {audio_sec:.1f}s\nASR: {t_asr:.2f}s\n"
                       f"CACHE HIT (similarity {sim:.2f}), lookup {t_cache*1000:.0f} ms: no LLM, no TTS\n"
                       f"time to first audio after you stop speaking: {t_asr + t_cache:.2f}s\n"
                       f"(not included: endpoint wait, browser upload, playback buffering)")
            return text, hit["answer"], (24000, hit["audio"]), timings

    answer, audio, t_first, t_llm_total, t_tts_first = answer_and_audio(text)
    if not answer:
        return text, "", None, "empty answer from the LLM"
    audio_i16 = to_i16(audio)
    if use_cache:
        cache.put(text, answer, audio_i16)
    est = t_asr + t_first + t_tts_first
    timings = (f"your speech: {audio_sec:.1f}s\nASR: {t_asr:.2f}s\n"
               f"LLM first clause (retrieval included): {t_first:.2f}s (whole answer {t_llm_total:.2f}s)\n"
               f"TTS first chunk: {t_tts_first:.2f}s\n"
               f"ESTIMATED time to first audio after you stop speaking: {est:.2f}s\n"
               f"(not included: endpoint wait, browser upload, playback buffering)")
    return text, answer, (24000, audio_i16), timings

with gr.Blocks() as demo:
    gr.Markdown("## Amharic bank voice assistant: speak, then press Stop")
    mic = gr.Audio(sources=["microphone", "upload"], type="filepath", label="Microphone")
    use_cache = gr.Checkbox(value=True, label="Use response cache")
    box_asr = gr.Textbox(label="You said (ASR)")
    box_ans = gr.Textbox(label="Assistant answer")
    out = gr.Audio(label="Spoken answer", type="numpy", autoplay=True)
    box_t = gr.Textbox(label="Timings", lines=6)
    mic.stop_recording(run_pipeline, [mic, use_cache], [box_asr, box_ans, out, box_t])

# password: from a Kaggle secret named DEMO_PASSWORD if you add one, otherwise a random one
try:
    from kaggle_secrets import UserSecretsClient
    DEMO_PASSWORD = UserSecretsClient().get_secret("DEMO_PASSWORD")
except Exception:
    DEMO_PASSWORD = secrets.token_urlsafe(8)
print("Demo login -> user: demo | password:", DEMO_PASSWORD)
demo.launch(share=True, auth=("demo", DEMO_PASSWORD))

Closing server running on port: 7860
Demo login -> user: demo | password: 123456
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6ac97a5ddd638788e2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Chunks: 100%|██████████| 1/1 [00:00<00:00, 23.86it/s]
